# 03 - Deteccao de Anomalias em Precos

Detecta desvios anormais em series de preco usando **Z-score em janela deslizante**
e **Isolation Forest**. Gera sinais de decisao: `buy`, `sell` ou `hold`.

**Decisao gerada:** *comprar, vender ou aguardar?*

In [ ]:
import sys
sys.path.append('../src')

from data_fetcher import get_historical_prices
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.ensemble import IsolationForest

In [ ]:
TICKER    = 'AAPL'
PERIODO   = '2y'
JANELA    = 30   # dias para Z-score
Z_THRESH  = 2.5  # limiar de anomalia

df = get_historical_prices([TICKER], period=PERIODO)
precos = df[TICKER].dropna()
print(f'Dados carregados: {len(precos)} dias')

In [ ]:
# --- Z-score em janela deslizante ---
media_roll  = precos.rolling(JANELA).mean()
std_roll    = precos.rolling(JANELA).std()
z_score     = (precos - media_roll) / std_roll

anomalia_z = z_score.abs() > Z_THRESH
print(f'Anomalias detectadas (Z-score): {anomalia_z.sum()}')

In [ ]:
# --- Isolation Forest ---
X = precos.values.reshape(-1, 1)
iso = IsolationForest(contamination=0.05, random_state=42)
labels = iso.fit_predict(X)  # -1 = anomalia, 1 = normal
anomalia_if = pd.Series(labels == -1, index=precos.index)
print(f'Anomalias detectadas (Isolation Forest): {anomalia_if.sum()}')

In [ ]:
# --- Geracao de Sinais ---
# Combina os dois metodos: anomalia confirmada em ambos
anomalia_consenso = anomalia_z & anomalia_if

sinal = pd.Series('hold', index=precos.index)
sinal[anomalia_consenso & (z_score < 0)] = 'buy'   # queda anormal
sinal[anomalia_consenso & (z_score > 0)] = 'sell'  # alta anormal

print(sinal.value_counts())

In [ ]:
# --- Visualizacao ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Preco com sinais
axes[0].plot(precos.index, precos.values, label='Preco', lw=1.2)
axes[0].scatter(precos[sinal == 'buy'].index,  precos[sinal == 'buy'],  color='green', marker='^', s=80, label='BUY',  zorder=5)
axes[0].scatter(precos[sinal == 'sell'].index, precos[sinal == 'sell'], color='red',   marker='v', s=80, label='SELL', zorder=5)
axes[0].set_title(f'{TICKER} - Preco e Sinais de Anomalia')
axes[0].legend()
axes[0].set_ylabel('Preco (USD)')

# Z-score
axes[1].plot(z_score.index, z_score.values, color='purple', lw=1, label='Z-score')
axes[1].axhline( Z_THRESH, color='red',   ls='--', lw=0.8)
axes[1].axhline(-Z_THRESH, color='green', ls='--', lw=0.8)
axes[1].axhline(0, color='gray', ls='-', lw=0.5)
axes[1].set_title('Z-score (janela 30 dias)')
axes[1].set_ylabel('Z-score')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.show()